Step 1: Install, bootstrap instructions

In [ ]:
!pip install --quiet anthropic pydantic
!rm -rf /content/astra-swarm 2>/dev/null
!git clone --depth 1 -q https://github.com/phdeore/astra-swarm.git /content/astra-swarm

import sys, os
sys.path.insert(0, "/content/astra-swarm/src")

from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()  # .strip() from earlier lesson

# Sanity import
from astra_swarm.alerts import (
    triage_chain, generate_synthetic_alerts,
    parse_alert, summarize, assess_severity,
)
print("ready")

Step 2: Generate 5 synthetic alerts to regenerate them later.

In [ ]:
import json
from pathlib import Path

alerts = generate_synthetic_alerts(5)

fixture_path = Path("/content/astra-swarm/data/synthetic/02_alerts.json")
fixture_path.parent.mkdir(parents=True, exist_ok=True)
fixture_path.write_text(json.dumps(alerts, indent=2))

for i, a in enumerate(alerts, 1):
    print(f"--- Alert {i} ---")
    print(a)
    print()

# Use the following to regenerate alerts.
# alerts = json.loads(fixture_path.read_text())

Step 3: Run the chain on one alert

In [ ]:
raw = alerts[0]

print("=== RAW ===")
print(raw)
print()

parsed = parse_alert(raw)
print("=== PARSED ===")
print(json.dumps(parsed, indent=2))
print()

summary = summarize(parsed)
print("=== SUMMARY ===")
print(summary)
print()

verdict = assess_severity(parsed, summary)
print("=== VERDICT ===")
print(json.dumps(verdict, indent=2))

Step 4: Run the whole chain on all 5 alerts

In [ ]:
results = [triage_chain(a) for a in alerts]

print(f"{'#':<3} {'severity':<10} {'conf':<6} description")
print("-" * 80)
for i, r in enumerate(results, 1):
    v = r["verdict"]
    desc = r["parsed"].get("description", "")[:60]      # Truncated description for display
    print(f"{i:<3} {v.get('severity',''):<10} {v.get('confidence', 0):<6.2f} {desc}")